<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/02_machine_learning/reinforcement_learning/experiment_ppo_actor_critic_policy_optimization_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
# from actor_critic import ActorCritic

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        # Placeholder for actual model layers
        self.actor_head = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )
        self.critic_head = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        action_probs = torch.softmax(self.actor_head(x), dim=-1)
        state_value = self.critic_head(x)
        return action_probs, state_value


class PPOAgent:
    def __init__(self, state_dim, action_dim):
        self.model = ActorCritic(state_dim, action_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)

        self.gamma = 0.99
        self.epsilon = 0.2

    def compute_advantage(self, rewards, values):
        advantages = []
        for r, v in zip(rewards, values):
            advantages.append(r - v)
        return advantages

    def update(self, states, actions, rewards, old_probs):
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions)
        rewards = torch.tensor(rewards, dtype=torch.float32)

        probs, values = self.model(states)
        values = values.squeeze()

        advantages = rewards - values.detach()

        new_probs = probs.gather(1, actions.unsqueeze(1)).squeeze()
        ratio = new_probs / old_probs

        clipped = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon)

        loss = -torch.min(ratio * advantages, clipped * advantages).mean()

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
